# S6E10 — 01 Baseline model

> **Pipeline position:** `00_eda` → **01 baseline** → `02_features_experiments` → `03_final_features` → …

| | |
|---|---|
| **Goal** | Establish a solid, untuned LightGBM baseline with cross-validation — the reference score that every later idea must beat. |
| **Metric** | ROC-AUC (probability that a random satisfied passenger gets a higher score than a random unsatisfied one; 0.5 = coin flip, 1.0 = perfect) |
| **Input** | `data/raw/train.csv`, `test.csv`, `sample_submission.csv` |
| **Output** | `submissions/submission_lgbm_baseline.csv` (first Kaggle submission) |
| **Runtime** | ~2 min on CPU |
| **Result** | **CV AUC 0.95885** (folds 0.9590 / 0.9578 / 0.9595 / 0.9588 / 0.9591) |

## Table of Contents
1. Setup & Data Loading
2. Preprocessing
3. Cross-Validation
4. Final Model & Submission
5. Results & decisions

> **Glossary:** *Cross-validation (CV)* = split the training data into 5 parts; train on 4, score on the 5th, repeat 5 times. *OOF predictions* = the 5 held-out parts put together: one honest prediction for every training row. *Early stopping* = stop adding trees when the validation score stops improving.

## 1. Setup & Data Loading
Imports, paths (Kaggle or local), and loading. Column names are lower-cased with underscores so they are easy to type and consistent across all notebooks.

In [1]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

# Paths: Kaggle input if we run on Kaggle; locally the project root is found automatically
# (works when Jupyter starts in the project root or in a subfolder).
KAGGLE_DIR = Path('/kaggle/input/competitions/playground-series-s6e10')
if KAGGLE_DIR.exists():
    DATA_DIR, PROC = KAGGLE_DIR, Path('/kaggle/working')
else:
    PROJECT = next(p for p in (Path('.'), Path('..')) if (p / 'data' / 'raw' / 'train.csv').exists())
    DATA_DIR, PROC = PROJECT / 'data' / 'raw', PROJECT / 'data' / 'processed'
SUBMISSIONS = Path('/kaggle/working') if KAGGLE_DIR.exists() else PROJECT / 'submissions'

print("Imports OK")
print(f"Data path: {DATA_DIR}")

Imports OK
Data path: data\raw


In [2]:
train = pd.read_csv(DATA_DIR / 'train.csv')
test  = pd.read_csv(DATA_DIR / 'test.csv')

train.columns = train.columns.str.lower().str.replace(' ', '_')
test.columns  = test.columns.str.lower().str.replace(' ', '_')

print(f"Train: {train.shape}")
print(f"Test:  {test.shape}")

Train: (699635, 23)
Test:  (299844, 22)


## 2. Preprocessing
Minimal on purpose — features are the subject of the next notebooks.
- **Missing arrival delay** (292 rows, a random pattern found in EDA): fill with the departure delay and add a flag `arrival_delay_missing`.
- **Drop `arrival_delay_in_minutes`:** correlation with departure delay is 0.84 (EDA §7): it carries almost the same information.
- **Categoricals** (`gender`, `customer_type`, `type_of_travel`, `class`) → pandas `category`: LightGBM splits on them natively, no one-hot needed.

In [3]:
# NaN flag — before fillna, while the signal is still present
train['arrival_delay_missing'] = train['arrival_delay_in_minutes'].isna().astype(int)
test['arrival_delay_missing']  = test['arrival_delay_in_minutes'].isna().astype(int)

# Only 292 rows have a missing arrival delay, so the fill rule hardly matters; the flag above preserves the information (EDA §3).
train['arrival_delay_in_minutes'] = train['arrival_delay_in_minutes'].fillna(
    train['departure_delay_in_minutes']
)
test['arrival_delay_in_minutes'] = test['arrival_delay_in_minutes'].fillna(
    test['departure_delay_in_minutes']
)

# Target
y = train['satisfaction'].astype(int)

# Features: drop id, target, arrival_delay (largely repeats departure delay, corr 0.84)
DROP_COLS = ['id', 'satisfaction', 'arrival_delay_in_minutes']

# Categorical features — LightGBM handles them natively
CAT_COLS = ['gender', 'customer_type', 'type_of_travel', 'class']

X = train.drop(columns=DROP_COLS)
X_test = test.drop(columns=['id', 'arrival_delay_in_minutes'])

# Convert categoricals to category dtype — required by LightGBM
for col in CAT_COLS:
    X[col]      = X[col].astype('category')
    X_test[col] = X_test[col].astype('category')

print(f"X shape: {X.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"\nFeatures ({len(X.columns)}):")
print(list(X.columns))

X shape: (699635, 21)
X_test shape: (299844, 21)

Features (21):
['age', 'flight_distance', 'inflight_wifi_service', 'departure/arrival_time_convenient', 'ease_of_online_booking', 'gate_location', 'food_and_drink', 'online_boarding', 'seat_comfort', 'inflight_entertainment', 'on-board_service', 'leg_room_service', 'baggage_handling', 'checkin_service', 'cleanliness', 'departure_delay_in_minutes', 'gender', 'customer_type', 'type_of_travel', 'class', 'arrival_delay_missing']


## 3. Cross-Validation
Reasonable fixed parameters (no tuning) and a stratified 5-fold split: each fold keeps the same satisfied/unsatisfied share (≈ 44% satisfied).

In [4]:
LGBM_PARAMS = {
    'objective':        'binary',
    'metric':           'auc',
    'learning_rate':    0.05,
    # 127 leaves: fairly flexible trees; lr 0.05 + early stopping keeps training short. Untuned on purpose: this is the REFERENCE every later idea must beat.
    'num_leaves':       127,
    'min_child_samples': 20,
    'subsample':        0.8,
    'colsample_bytree': 0.8,
    'n_estimators':     1000,
    'early_stopping_rounds': 50,
    'verbose':          -1,
    'random_state':     42,
}

In [5]:
N_FOLDS = 5
# Fixed 5-fold split (seed 42). Reused in every later notebook, which is what makes all scores comparable and the models stackable.
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

oof_preds  = np.zeros(len(X))          # OOF predictions on train
test_preds = np.zeros(len(X_test))     # averaged predictions on test
fold_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx],  y.iloc[val_idx]

    model = lgb.LGBMClassifier(**LGBM_PARAMS)
    # Early stopping watches the validation fold, which makes fold scores very slightly optimistic. Standard Kaggle practice, identical for every model, so comparisons stay fair.
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        categorical_feature=CAT_COLS,
    )

    val_pred  = model.predict_proba(X_val)[:, 1]
    oof_preds[val_idx] = val_pred

    test_preds += model.predict_proba(X_test)[:, 1] / N_FOLDS

    score = roc_auc_score(y_val, val_pred)
    fold_scores.append(score)
    print(f"Fold {fold} AUC: {score:.5f}")

# Score on ALL out-of-fold predictions together: the number we compare across the whole project.
cv_score = roc_auc_score(y, oof_preds)
print(f"\nCV AUC (OOF): {cv_score:.5f}")
print(f"Mean fold AUC: {np.mean(fold_scores):.5f} ± {np.std(fold_scores):.5f}")

Fold 1 AUC: 0.95900


Fold 2 AUC: 0.95780


Fold 3 AUC: 0.95954


Fold 4 AUC: 0.95878


Fold 5 AUC: 0.95910

CV AUC (OOF): 0.95885
Mean fold AUC: 0.95885 ± 0.00058


## 4. Final Model & Submission
Test predictions are the **average of the 5 fold models** (no retraining on 100% of the data). Each fold model saw 80% of train; averaging five of them is stable and gives the same-quality prediction for test.

In [6]:
sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')
sub['satisfaction'] = test_preds                      # probability of "satisfied"; ROC-AUC only needs the ranking
SUBMISSIONS.mkdir(exist_ok=True)
sub.to_csv(SUBMISSIONS / 'submission_lgbm_baseline.csv', index=False)

print(f"Submission saved. CV AUC: {cv_score:.5f}")
sub.head()

Submission saved. CV AUC: 0.95885


,id,satisfaction
0,699635,0.266677
1,699636,0.870664
2,699637,0.926266
3,699638,0.062872
4,699639,0.045627


## 5. Results & decisions
- **Baseline CV AUC = 0.95885** (std over folds 0.0006; the first run on the Mac gave 0.95882, the gap is multithreading noise). Everything later is measured against this number.
- **Fold 2 is consistently the hardest** (0.9578 here); the same fold is the weakest for every later model too, so it reflects *that split*, not model quality. This is why we compare models fold by fold on the identical split.
- **Differences below ≈ 0.0003 are noise** (fold-to-fold spread and public-LB noise), so a new idea must clear a higher bar before we keep it.
- **Next:** `02_features_experiments` tests feature ideas one at a time against this baseline.